# 🧬 Transformers & Attention: Zero to Hero — A Guided Lab

Every modern LLM (GPT, Claude, Gemini) is built on the **Transformer** architecture, and the
Transformer's core idea is **attention**. This lab builds attention and a mini-Transformer
**from raw NumPy math** — no black boxes — so you understand exactly what happens inside an LLM
when it reads your prompt.

**Beginner-first.** Every chapter explains the *concept* and the *math* in plain language before
any code. If you've done the NumPy and Neural Networks labs, you're ready.

**How this lab works** — 📖 Theory (detailed) → 🧠 Mental model → 🖼️ ASCII diagram →
🔬 Worked example (by-hand math, then code) → ⚡ Pro tips → ⚠️ Traps → ✏️ Your Turn → ✅ Solution.

**Roadmap**
1. Why attention? The problem with older sequence models
2. Tokenization & embeddings
3. Positional encoding
4. The attention mechanism, step by step
5. Scaled dot-product attention (the formula)
6. Multi-head attention
7. The Transformer block (residuals, layer norm, feed-forward)
8. Causal masking (why GPT can't peek ahead)
9. From logits to text: the generation loop
10. 🏆 Capstone: a tiny GPT-style model that generates text


In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
print("Ready. We'll build everything with plain NumPy so every step is visible.")

---
## Chapter 1 — Why Attention? The Problem With Older Models

📖 **Theory.** Before Transformers, models like RNNs read a sentence **one word at a time**,
carrying a single "memory" (hidden state) forward. Problem: by the time an RNN reaches word 50,
it has often "forgotten" important details from word 2 — the memory is a bottleneck squeezed
through one fixed-size vector.

**Attention** fixes this: instead of relying on one shrinking memory, every word can **directly
look at every other word** in the sentence and decide how much to "pay attention" to it. No
information has to survive a long relay race.

🖼️ **Diagram — RNN bottleneck vs. attention**
```
 RNN:      word1 -> [mem] -> word2 -> [mem] -> word3 -> [mem] -> ... -> word50
                      (each step, old info gets diluted/forgotten)

 Attention: word1 ◄──────────────┐
            word2 ◄───────┐      │   every word can look directly
            word3 ◄────┐  │      │   at every other word, at once
            ...         └──┴──────┘
```

🧠 **Mental model.** Imagine reading a sentence and, for each word, glancing back at *every*
other word to decide which ones matter for understanding this one. That glancing-and-weighing
is literally what attention computes as numbers.


In [ ]:
# A concrete motivating example: pronoun resolution needs long-range attention.
sentence = ["The", "cat", "sat", "on", "the", "mat", "because", "it", "was", "tired"]
# "it" (index 7) refers to "cat" (index 1) -- far apart in the sequence.
# A good attention mechanism should let "it" attend strongly to "cat".
print("Sentence:", " ".join(sentence))
print(f"'it' is at position {sentence.index('it')}, 'cat' is at position {sentence.index('cat')}")
print("Distance apart:", sentence.index('it') - sentence.index('cat'), "words -- attention handles this directly.")

### ✏️ Your Turn 1.1
In a comment, explain in your own words why a fixed-size "memory" bottleneck makes long-range
dependencies (like "it" referring back to "cat") hard for older models.

In [ ]:
# your explanation here


✅ **Solution**
```python
# A fixed-size hidden state must compress everything seen so far into one vector.
# As the sentence grows, early details get overwritten/diluted by later ones --
# so information from far away (like "cat") may not survive to influence "it".
```

---
## Chapter 2 — Tokenization & Embeddings

📖 **Theory.** A Transformer can't read text directly — text must become numbers.
1. **Tokenization** splits text into pieces (tokens) — words, sub-words, or characters — and
   maps each to an integer ID from a fixed vocabulary.
2. **Embedding** looks up each ID in a table of learned vectors — turning a discrete ID into a
   dense vector that captures meaning (similar words → similar vectors, as you saw in the
   Embeddings & Search lab).

🖼️ **Diagram — text to vectors**
```
 "the cat sat" ─tokenize─► [4, 19, 7] ─embed─► [[0.1,-0.3,...],   (each row = one
                          (integer IDs)         [0.5, 0.2,...],    token's vector,
                                                 [-0.2,0.4,...]]   dim = e.g. 512)
```

🧠 **Mental model.** The embedding table is a big lookup dictionary: `token_id -> vector`. It
starts random and is *learned* during training so that semantically similar tokens end up
nearby in vector space — exactly like the embeddings you built in the Search lab.


In [ ]:
# A tiny toy vocabulary and a random (untrained) embedding table
vocab = ["the","cat","sat","on","mat","because","it","was","tired","dog"]
vocab_size = len(vocab)
d_model = 8   # embedding dimension (real models use 512-12000+)

np.random.seed(0)
embedding_table = np.random.randn(vocab_size, d_model) * 0.1
token_to_id = {w:i for i,w in enumerate(vocab)}

def tokenize(sentence_words):
    return [token_to_id[w] for w in sentence_words]

def embed(token_ids):
    return embedding_table[token_ids]   # fancy indexing: pick a row per id

sent = ["the","cat","sat"]
ids = tokenize(sent)
vecs = embed(ids)
print("token ids:", ids)
print("embedded shape:", vecs.shape, "(3 tokens x 8 dims)")
print(vecs)

⚡ **Pro tip.** Real tokenizers (like BPE — Byte Pair Encoding) split *sub-words*, not just
whole words, so rare words like "internationalization" become a few common pieces. This keeps
the vocabulary small while still covering any input text.

### ✏️ Your Turn 2.1
Tokenize and embed the sentence `["cat", "was", "tired"]`. Confirm the output shape is `(3, 8)`.

In [ ]:
s2 = ["cat","was","tired"]
ids2 = None
vecs2 = None
print(vecs2.shape if vecs2 is not None else None)

✅ **Solution**
```python
ids2 = tokenize(s2)
vecs2 = embed(ids2)
print(vecs2.shape)   # (3, 8)
```

---
## Chapter 3 — Positional Encoding

📖 **Theory.** Attention (next chapter) looks at *all* words at once, with no built-in sense of
**order** — "cat sat mat" and "mat sat cat" would look identical without help. **Positional
encoding** adds a unique pattern per position to each embedding, so the model can tell word
order apart.

The original Transformer paper uses sine/cosine waves of different frequencies:

```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

🖼️ **Diagram — adding position info**
```
 word embedding:      [0.1, -0.3, 0.5, ...]     (meaning, no order info)
      +
 positional encoding: [0.0,  1.0, 0.0, ...]     (unique wave pattern per position)
      =
 final input:         [0.1,  0.7, 0.5, ...]     (meaning + position, combined by addition)
```

🧠 **Mental model.** Each position gets its own unique "fingerprint" pattern (a mix of waves at
different frequencies), added directly onto the word's embedding — like stamping a timestamp
onto each word before it enters the model.


In [ ]:
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]                  # (seq_len, 1)
    i = np.arange(d_model)[None, :]                     # (1, d_model)
    angle_rates = 1 / np.power(10000, (2*(i//2)) / d_model)
    angles = pos * angle_rates
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angles[:, 0::2])   # even dims: sine
    pe[:, 1::2] = np.cos(angles[:, 1::2])   # odd dims: cosine
    return pe

pe = positional_encoding(seq_len=3, d_model=8)
print("positional encoding (3 positions x 8 dims):\n", pe)

# combine with our word embeddings from Chapter 2
final_input = vecs + pe
print("\nword embedding + position -> model input:\n", final_input)

⚠️ **Common trap.** Forgetting positional encoding entirely makes attention **permutation-
invariant** — the model literally cannot tell "dog bites man" from "man bites dog". This is a
famous, easy-to-miss bug when implementing attention from scratch.

### ✏️ Your Turn 3.1
Generate positional encodings for a sequence of length 5 with `d_model=8`. Confirm positions 0
and 1 have different patterns (they should — same word at different positions must look different).

In [ ]:
pe5 = None
positions_differ = None
print(positions_differ)

✅ **Solution**
```python
pe5 = positional_encoding(5, 8)
positions_differ = not np.allclose(pe5[0], pe5[1])   # True
```

---
## Chapter 4 — The Attention Mechanism, Step by Step

📖 **Theory.** For each word, attention asks: **"which other words should I gather information
from, and how much?"** It does this with three learned projections of each word's vector:
- **Query (Q)** — "what am I looking for?" (this word's question)
- **Key (K)** — "what do I contain?" (every word's advertisement of its content)
- **Value (V)** — "what information do I actually offer?" (every word's payload)

A word's Query is compared against every word's Key to get **attention scores**; those scores
weight how much of each word's Value gets mixed into the output.

🖼️ **Diagram — Q, K, V roles**
```
 word "it":     Query  = "who am I referring to?"
 word "cat":    Key    = "I am an animal, a subject"      } compare Query·Key -> score
 word "mat":    Key    = "I am an object, a location"     }
                Values = the actual content each word contributes if selected
```

🧠 **Mental model.** Think of a library search: your **Query** is your search terms, each book's
**Key** is its index card, and matching them tells you which books (their **Values**) to
actually read and how much weight to give each.


In [ ]:
d_model = 8
d_k = 8   # dimension of queries/keys (often smaller than d_model in real models; kept equal here for simplicity)

np.random.seed(1)
W_Q = np.random.randn(d_model, d_k) * 0.3
W_K = np.random.randn(d_model, d_k) * 0.3
W_V = np.random.randn(d_model, d_k) * 0.3

X = final_input   # our 3-token embedded+positioned input from Chapter 3, shape (3,8)

Q = X @ W_Q   # (3,8) @ (8,8) -> (3,8): one query vector per token
K_ = X @ W_K  # one key vector per token
V = X @ W_V   # one value vector per token

print("Q shape:", Q.shape, "-- one row per word, this word's 'question'")
print("K shape:", K_.shape, "-- one row per word, this word's 'advertisement'")
print("V shape:", V.shape, "-- one row per word, this word's 'content'")

### ✏️ Your Turn 4.1
`W_Q`, `W_K`, `W_V` are **learned** during training (random here, since we haven't trained
anything). In a comment, explain what it would mean for the model to "learn" a good `W_Q`.

In [ ]:
# your explanation here


✅ **Solution**
```python
# Learning W_Q means adjusting it (via gradient descent, like in the PyTorch lab) so that
# the resulting Query vectors end up matching the RIGHT Keys for the task -- e.g. so "it"'s
# query vector aligns strongly with "cat"'s key vector, and weakly with "mat"'s.
```

---
## Chapter 5 — Scaled Dot-Product Attention (the formula)

📖 **Theory.** The full attention formula:

```
Attention(Q, K, V) = softmax( Q Kᵀ / √d_k ) V
```

Step by step:
1. **QKᵀ** — dot product of every Query with every Key → a matrix of raw similarity scores.
2. **/√d_k** — scale down (prevents scores from growing huge and destabilizing softmax).
3. **softmax** — turn scores into a probability distribution per row (sums to 1) → attention weights.
4. **× V** — use those weights to compute a weighted average of Values → the output.

🖼️ **Diagram — the four steps**
```
 Q Kᵀ          scores matrix          softmax           weighted sum
 (3x8)@(8x3) → (3x3) raw scores  →  (3x3) weights   →  weights @ V → (3x8) output
              "how much does word    (each row sums      "the actual info
               i relate to word j?"   to 1.0)              word i gathers"
```


In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)   # numerical stability trick
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)     # (seq_len, seq_len) raw similarity
    weights = softmax(scores, axis=-1)  # each row -> probability distribution
    output = weights @ V                # (seq_len, d_model) weighted combination
    return output, weights

output, attn_weights = scaled_dot_product_attention(Q, K_, V)
print("attention weights (each row sums to 1):\n", attn_weights)
print("\nrow sums (should all be 1.0):", attn_weights.sum(axis=1))
print("\noutput shape:", output.shape)

⚠️ **Common trap.** Skipping the `/√d_k` scaling makes dot products grow large for bigger
dimensions, pushing softmax into a near one-hot ("all-or-nothing") distribution — this stalls
learning (vanishing gradients through softmax). Always scale.

### ✏️ Your Turn 5.1
Compute attention **without** the `/√d_k` scaling (just `softmax(Q@K.T) @ V`) and compare the
resulting attention weights to the scaled version. Are they more "peaked" (closer to one-hot)?

In [ ]:
unscaled_scores = None
unscaled_weights = None
print(unscaled_weights)

✅ **Solution**
```python
unscaled_scores = Q @ K_.T          # no /sqrt(d_k)
unscaled_weights = softmax(unscaled_scores, axis=-1)
# Compare: unscaled weights are typically more extreme (closer to 0 or 1) than scaled ones.
```

---
## Chapter 6 — Multi-Head Attention

📖 **Theory.** One attention computation can only capture **one kind** of relationship (e.g.
"who does this pronoun refer to"). **Multi-head attention** runs several attention computations
**in parallel**, each with its own learned `W_Q, W_K, W_V` — so different heads can specialize:
one might track grammar, another might track topic, etc. Their outputs are concatenated and
mixed back together.

🖼️ **Diagram — multiple heads in parallel**
```
        ┌─► head 1: Attention(Q1,K1,V1) ─► output1 ┐
 input ─┼─► head 2: Attention(Q2,K2,V2) ─► output2 ─┼─► concat ─► linear mix ─► final output
        └─► head 3: Attention(Q3,K3,V3) ─► output3 ┘
```

🧠 **Mental model.** Like having several analysts read the same sentence, each looking for a
different pattern (grammar, topic, coreference), then combining their notes into one report.


In [ ]:
def multi_head_attention(X, num_heads, d_model):
    assert d_model % num_heads == 0
    d_head = d_model // num_heads
    outputs = []
    for h in range(num_heads):
        np.random.seed(100 + h)   # a different random projection per head (stand-in for learned weights)
        Wq = np.random.randn(d_model, d_head) * 0.3
        Wk = np.random.randn(d_model, d_head) * 0.3
        Wv = np.random.randn(d_model, d_head) * 0.3
        Qh, Kh, Vh = X @ Wq, X @ Wk, X @ Wv
        out_h, _ = scaled_dot_product_attention(Qh, Kh, Vh)
        outputs.append(out_h)
    concatenated = np.concatenate(outputs, axis=-1)   # (seq_len, num_heads*d_head) == (seq_len, d_model)
    np.random.seed(999)
    W_O = np.random.randn(d_model, d_model) * 0.3      # final mixing layer
    return concatenated @ W_O

mha_out = multi_head_attention(final_input, num_heads=2, d_model=8)
print("multi-head attention output shape:", mha_out.shape)   # same shape as input -- stackable!
print(mha_out)

⚡ **Pro tip.** Notice the output shape **matches the input shape** `(seq_len, d_model)`. This
is intentional — it lets you **stack multiple Transformer blocks** on top of each other, each
one refining the representation further (real models stack dozens of these).

### ✏️ Your Turn 6.1
Run `multi_head_attention` with `num_heads=4` instead of 2 (still `d_model=8`, so each head gets
dimension 2). Confirm the output shape is unchanged.

In [ ]:
mha_out4 = None
print(mha_out4.shape if mha_out4 is not None else None)

✅ **Solution**
```python
mha_out4 = multi_head_attention(final_input, num_heads=4, d_model=8)
print(mha_out4.shape)   # still (3, 8) -- same as input
```

---
## Chapter 7 — The Transformer Block (residuals, layer norm, feed-forward)

📖 **Theory.** A full Transformer block wraps attention with three more ingredients:
- **Residual connections** (`x + sublayer(x)`) — let gradients flow directly through, avoiding
  vanishing gradients in deep stacks (same idea as in the Neural Networks lab).
- **Layer normalization** — rescales each token's vector to stable mean/variance, keeping
  training stable.
- **Feed-forward network** — a small 2-layer MLP applied to *each token independently*, adding
  extra representational power after attention mixes information *across* tokens.

🖼️ **Diagram — one Transformer block**
```
 x ──► Multi-Head Attention ──► (+x, residual) ──► LayerNorm ──► FeedForward ──► (+prev, residual) ──► LayerNorm ──► out
```


In [ ]:
def layer_norm(x, eps=1e-6):
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)

def feed_forward(x, d_model, d_ff=16):
    np.random.seed(42)
    W1 = np.random.randn(d_model, d_ff) * 0.3
    W2 = np.random.randn(d_ff, d_model) * 0.3
    hidden = np.maximum(0, x @ W1)   # ReLU
    return hidden @ W2

def transformer_block(x, num_heads, d_model):
    attn_out = multi_head_attention(x, num_heads, d_model)
    x = layer_norm(x + attn_out)             # residual + norm
    ff_out = feed_forward(x, d_model)
    x = layer_norm(x + ff_out)                # residual + norm
    return x

block_out = transformer_block(final_input, num_heads=2, d_model=8)
print("transformer block output shape:", block_out.shape)
print(block_out)

⚠️ **Common trap.** Without residual connections, stacking many blocks makes gradients vanish
during backprop (the same problem you saw with deep RNNs) — the model becomes very hard to
train. Residuals are not optional polish; they're structurally necessary for depth.

### ✏️ Your Turn 7.1
Stack **two** `transformer_block` calls in sequence (feed the output of the first into the
second). Confirm the final shape is still `(3, 8)`.

In [ ]:
stacked = None
print(stacked.shape if stacked is not None else None)

✅ **Solution**
```python
b1 = transformer_block(final_input, num_heads=2, d_model=8)
stacked = transformer_block(b1, num_heads=2, d_model=8)
print(stacked.shape)   # (3, 8)
```

---
## Chapter 8 — Causal Masking (why GPT can't peek ahead)

📖 **Theory.** When *generating* text, a model must predict the next word using only **past**
words — it can't be allowed to "cheat" by looking at future words it hasn't generated yet.
**Causal masking** enforces this: before softmax, we set the attention score for any
"future" position to **-infinity**, so softmax turns it into 0 — zero attention to the future.

🖼️ **Diagram — the causal mask**
```
          the  cat  sat          (columns = keys/positions being attended TO)
 the    [  ✓    ✗    ✗  ]        the row "the" may only see itself
 cat    [  ✓    ✓    ✗  ]        "cat" may see "the" and itself, not "sat"
 sat    [  ✓    ✓    ✓  ]        "sat" may see everything before + itself
       (rows = queries/positions doing the attending)
```


In [ ]:
def causal_mask(seq_len):
    # upper triangle (excluding diagonal) = disallowed future positions
    mask = np.triu(np.ones((seq_len, seq_len)), k=1)
    return mask   # 1 = masked (block), 0 = allowed

def causal_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    mask = causal_mask(Q.shape[0])
    scores = np.where(mask == 1, -1e9, scores)   # -infinity -> softmax gives ~0
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

out, causal_weights = causal_attention(Q, K_, V)
print("causal attention weights (upper triangle should be ~0):\n", causal_weights.round(3))

⚡ **Pro tip.** This is exactly the difference between **GPT-style (decoder-only, causal)**
models used for generation and **BERT-style (encoder, bidirectional)** models used for
understanding — BERT has no causal mask because it isn't generating text left-to-right.

### ✏️ Your Turn 8.1
Confirm that in `causal_weights`, every value **above** the diagonal is (numerically) zero.

In [ ]:
all_future_blocked = None
print(all_future_blocked)

✅ **Solution**
```python
upper_triangle_values = causal_weights[np.triu_indices(3, k=1)]
all_future_blocked = np.allclose(upper_triangle_values, 0, atol=1e-6)
```

---
## Chapter 9 — From Logits to Text: the Generation Loop

📖 **Theory.** After all Transformer blocks, a final linear layer projects each token's vector
to **vocabulary-size logits** — one score per possible next word. Softmax turns logits into
probabilities; we sample (or take the max) to pick the next token, **append it**, and repeat.
This is the same autoregressive loop behind every GPT-style model, including real production LLMs.

🖼️ **Diagram — the generation loop**
```
 prompt tokens ─► [Transformer blocks] ─► logits (last position) ─► softmax ─► sample
      ▲                                                                          │
      └──────────────────── append sampled token, repeat ◄───────────────────────┘
```


In [ ]:
def output_projection(x, d_model, vocab_size):
    np.random.seed(7)
    W_out = np.random.randn(d_model, vocab_size) * 0.3
    return x @ W_out   # (seq_len, vocab_size) -- logits per position

def generate_next_token(token_ids, temperature=1.0):
    ids = np.array(token_ids)
    x = embed(ids) + positional_encoding(len(ids), d_model)
    x = transformer_block(x, num_heads=2, d_model=d_model)
    logits = output_projection(x, d_model, vocab_size)
    last_logits = logits[-1] / temperature      # only the LAST position predicts the next token
    probs = softmax(last_logits)
    next_id = np.random.choice(vocab_size, p=probs)
    return next_id, probs

np.random.seed(3)
prompt = tokenize(["the","cat"])
next_id, probs = generate_next_token(prompt)
print("prompt:", [vocab[i] for i in prompt])
print("predicted next token:", vocab[next_id])
print("top-3 probable next words:")
for i in np.argsort(probs)[-3:][::-1]:
    print(f"  {vocab[i]:10s} {probs[i]:.3f}")

### ✏️ Your Turn 9.1
Write a loop that starts from `["the"]` and generates **3 more tokens**, appending each one and
feeding the growing sequence back in (autoregressive generation).

In [ ]:
generated = tokenize(["the"])
# loop 3 times: predict next, append, repeat
for _ in range(3):
    pass
print([vocab[i] for i in generated])

✅ **Solution**
```python
generated = tokenize(["the"])
for _ in range(3):
    next_id, _ = generate_next_token(generated)
    generated.append(int(next_id))
print([vocab[i] for i in generated])
```

---
## 🏆 Chapter 10 — Capstone: A Tiny GPT-Style Text Generator

Combine everything into a `MiniGPT` class: embeddings + positional encoding → stacked causal
Transformer blocks → output projection → autoregressive generation. Build it before revealing
the solution — this is the architecture behind every LLM you've used in this course.

In [ ]:
# Your MiniGPT here
class MiniGPT:
    def __init__(self, vocab, d_model=8, num_heads=2, num_layers=2):
        pass
    def forward(self, token_ids):
        pass
    def generate(self, prompt_words, num_new_tokens=5, temperature=1.0):
        pass

# gpt = MiniGPT(vocab)
# print(gpt.generate(["the","cat"], num_new_tokens=4))


✅ **Capstone Solution**
```python
class MiniGPT:
    def __init__(self, vocab, d_model=8, num_heads=2, num_layers=2, seed=0):
        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_layers = num_layers
        rng = np.random.RandomState(seed)
        self.embedding_table = rng.randn(self.vocab_size, d_model) * 0.1
        self.token_to_id = {w: i for i, w in enumerate(vocab)}

    def _embed(self, ids):
        return self.embedding_table[ids] + positional_encoding(len(ids), self.d_model)

    def forward(self, token_ids):
        x = self._embed(np.array(token_ids))
        for _ in range(self.num_layers):
            attn_out, _ = causal_attention(x @ np.eye(self.d_model), x @ np.eye(self.d_model), x @ np.eye(self.d_model))
            x = layer_norm(x + attn_out)
            x = layer_norm(x + feed_forward(x, self.d_model))
        logits = output_projection(x, self.d_model, self.vocab_size)
        return logits

    def generate(self, prompt_words, num_new_tokens=5, temperature=1.0):
        ids = [self.token_to_id[w] for w in prompt_words]
        for _ in range(num_new_tokens):
            logits = self.forward(ids)
            probs = softmax(logits[-1] / temperature)
            next_id = np.random.choice(self.vocab_size, p=probs)
            ids.append(int(next_id))
        return [self.vocab[i] for i in ids]

gpt = MiniGPT(vocab)
np.random.seed(5)
print("Generated:", " ".join(gpt.generate(["the", "cat"], num_new_tokens=4)))
```

🎉 **You built a Transformer from scratch!** Tokenization, embeddings, positional encoding,
Q/K/V, scaled dot-product attention, multi-head attention, residuals + layer norm, causal
masking, and autoregressive generation — the exact architecture (at toy scale) behind GPT,
Claude, and every modern LLM. Real models differ only in **scale** (billions of parameters,
learned not random weights) and engineering, not in these core mechanics.

---
### 📌 Concept Quick-Reference
**Tokenize + embed:** text → IDs → learned dense vectors
**Positional encoding:** sin/cos waves added to embeddings so order isn't lost
**Q, K, V:** query="what I seek", key="what I offer", value="my actual content"
**Attention formula:** `softmax(QKᵀ/√d_k) V` — scaled similarity → weights → weighted sum
**Multi-head:** several attention computations in parallel, concatenated + mixed
**Transformer block:** attention → residual+norm → feed-forward → residual+norm
**Causal mask:** block future positions with -infinity before softmax (GPT-style generation)
**Generation loop:** predict next-token logits → softmax → sample → append → repeat
